# Flipkart E-commerce Sales Analysis
### Data Analyst Portfolio Project

**Objective:** Explore customer and product purchase behaviour, identify sales patterns, and communicate useful business insights with Python.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Key questions**
- What does the purchase-value distribution look like?
- Which customer segments have higher average purchase values?
- How do purchase values differ by city, age group, gender, and payment mode?
- Are discounts, delivery time, ratings, or product categories associated with purchase value?
- What data-quality issues should be addressed before deeper analysis?

> **Dataset note:** This notebook uses `train.csv` and optionally `test.csv`. Place them in the same folder as this notebook, or update `DATA_DIR` below. The analysis checks whether expected columns exist before using them.

## 1. Import libraries and configure the notebook

We use Pandas for data manipulation, NumPy for numerical work, and Matplotlib/Seaborn for charts. The plotting settings keep figures readable and consistent.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 30)

# Change this if your CSV files are stored in another folder.
DATA_DIR = Path(".")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"

## 2. Load the data

First, confirm the files exist, then inspect the dataset dimensions and a few rows. If the CSV files are stored in Google Colab, set `DATA_DIR = Path("/content")`.

In [ ]:
if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {TRAIN_FILE.resolve()}. "
        "Place train.csv beside this notebook or update DATA_DIR."
    )

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE) if TEST_FILE.exists() else None

print(f"Training data: {train.shape[0]:,} rows × {train.shape[1]:,} columns")
if test is not None:
    print(f"Test data:     {test.shape[0]:,} rows × {test.shape[1]:,} columns")
else:
    print("Optional test.csv not found; continuing with train.csv only.")

display(train.head())

## 3. Understand the dataset structure

Review column names, data types, and summary statistics before drawing conclusions. This helps identify numeric fields stored as text and unexpected values.

In [ ]:
print("Column names:")
print(train.columns.tolist())

print("\nData types and non-null counts:")
train.info()

display(train.describe(include="all").T)

## 4. Data quality checks

Check duplicate records, missing values, and the percentage of missing data in each column. Missing values are not automatically errors; decide how to handle them based on their meaning and the business question.

In [ ]:
duplicate_count = train.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,} ({duplicate_count / len(train):.2%} of rows)")

quality = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean().mul(100).round(2),
    "dtype": train.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

display(quality)

missing = quality.loc[quality["missing_count"] > 0, "missing_percent"]
if not missing.empty:
    ax = missing.sort_values().plot(kind="barh", figsize=(9, max(3, len(missing) * 0.35)))
    ax.set_title("Missing values by column")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 5. Define reusable column checks

The notebook you started assumes several specific columns. These helper lists make the analysis more robust if a column is absent or named differently.

In [ ]:
def available_columns(columns):
    return [col for col in columns if col in train.columns]

numeric_cols = train.select_dtypes(include=np.number).columns.tolist()
categorical_candidates = [
    "Gender", "Age", "City_Category", "Payment_Mode",
    "Marital_Status", "Is_Returned", "Product_Category_1"
]
categorical_cols = available_columns(categorical_candidates)

print("Numeric columns:", numeric_cols)
print("Available analysis categories:", categorical_cols)

## 6. Analyse purchase values

The histogram shows the shape of the purchase distribution; the boxplot highlights spread and potential extreme values. Outliers should be investigated rather than removed automatically.

In [ ]:
if "Purchase" in train.columns:
    purchase = pd.to_numeric(train["Purchase"], errors="coerce").dropna()
    display(purchase.describe().to_frame("Purchase"))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.histplot(purchase, bins=40, kde=True, ax=axes[0])
    axes[0].set_title("Distribution of Purchase Value")
    axes[0].set_xlabel("Purchase value")

    sns.boxplot(x=purchase, ax=axes[1])
    axes[1].set_title("Purchase Value: Potential Outliers")
    axes[1].set_xlabel("Purchase value")

    plt.tight_layout()
    plt.show()
else:
    print("The required column 'Purchase' is not available.")

## 7. Compare purchase value across customer segments

Use median and mean together where possible. The mean can be affected by unusually large purchases, while the median describes the middle observation.

In [ ]:
segment_cols = available_columns(["Gender", "Age", "City_Category", "Payment_Mode",
                                  "Marital_Status", "Is_Returned"])

if "Purchase" in train.columns and segment_cols:
    for col in segment_cols:
        summary = (
            train.groupby(col, dropna=False)["Purchase"]
            .agg(customers_or_rows="count", mean_purchase="mean", median_purchase="median")
            .sort_values("mean_purchase", ascending=False)
            .round(2)
        )
        print(f"\nPurchase summary by {col}:")
        display(summary)

        plt.figure(figsize=(9, 4.5))
        order = summary.index.astype(str).tolist()
        plot_data = train.copy()
        plot_data[col] = plot_data[col].astype(str)
        sns.boxplot(data=plot_data, x=col, y="Purchase", order=order)
        plt.title(f"Purchase Value by {col}")
        plt.xlabel(col.replace("_", " "))
        plt.ylabel("Purchase value")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("Need 'Purchase' and at least one available customer-segment column.")

## 8. Category counts

These charts show the number of records in key customer and transaction groups. Counts describe how frequently a group appears; they do not necessarily mean that group spends more.

In [ ]:
count_cols = available_columns(["Payment_Mode", "City_Category", "Age", "Gender"])

if count_cols:
    n = len(count_cols)
    fig, axes = plt.subplots(n, 1, figsize=(10, max(3.5, 3.2 * n)))
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, count_cols):
        counts = train[col].astype("string").fillna("Missing").value_counts()
        counts.plot(kind="bar", ax=ax)
        ax.set_title(f"Record Count by {col}")
        ax.set_xlabel(col.replace("_", " "))
        ax.set_ylabel("Number of rows")
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()
else:
    print("No expected category columns are available.")

## 9. Correlation between numeric fields

Correlation measures linear association between numeric columns. It does not prove causation, and correlations can be misleading when variables are coded as numbers but represent categories.

In [ ]:
corr_cols = train.select_dtypes(include=np.number).columns.tolist()
if len(corr_cols) >= 2:
    corr = train[corr_cols].corr()
    plt.figure(figsize=(max(8, 0.8 * len(corr_cols)), max(6, 0.65 * len(corr_cols))))
    sns.heatmap(corr, annot=len(corr_cols) <= 12, fmt=".2f",
                cmap="vlag", center=0, square=False)
    plt.title("Correlation Heatmap — Numeric Columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are needed for a correlation heatmap.")

## 10. Purchase value by age and city

A grouped summary can reveal which age/city combinations have higher average purchases. Review the row counts alongside averages so that small groups are not over-interpreted.

In [ ]:
if all(c in train.columns for c in ["Age", "City_Category", "Purchase"]):
    age_city = train.pivot_table(
        index="Age", columns="City_Category", values="Purchase",
        aggfunc=["mean", "count"], observed=False
    )
    display(age_city.round(2))

    mean_pivot = train.pivot_table(
        index="Age", columns="City_Category", values="Purchase",
        aggfunc="mean", observed=False
    )
    mean_pivot.plot(kind="bar", figsize=(10, 5))
    plt.title("Average Purchase by Age Group and City Category")
    plt.xlabel("Age group")
    plt.ylabel("Average purchase value")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()
else:
    print("Need Age, City_Category, and Purchase columns.")

## 11. Product and discount analysis

These plots help investigate potential relationships. A scatterplot can be noisy with large datasets, so a reproducible sample is used for display. Association does not establish that discounts cause higher purchases.

In [ ]:
if "Purchase" in train.columns and "Discount_Percent" in train.columns:
    sample_cols = ["Purchase", "Discount_Percent"]
    sample = train[sample_cols].dropna().sample(
        n=min(4000, train[sample_cols].dropna().shape[0]), random_state=42
    )
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=sample, x="Discount_Percent", y="Purchase", alpha=0.35)
    plt.title("Purchase Value vs Discount Percentage (Sample)")
    plt.tight_layout()
    plt.show()
else:
    print("Discount analysis skipped: Purchase and/or Discount_Percent is missing.")

if "Product_Category_1" in train.columns and "Purchase" in train.columns:
    product_summary = (
        train.groupby("Product_Category_1")["Purchase"]
        .agg(rows="count", mean_purchase="mean", median_purchase="median", total_purchase="sum")
        .sort_values("mean_purchase", ascending=False)
        .round(2)
    )
    display(product_summary)

    product_summary["mean_purchase"].sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title("Average Purchase by Product Category")
    plt.xlabel("Average purchase value")
    plt.ylabel("Product category")
    plt.tight_layout()
    plt.show()
else:
    print("Product-category analysis skipped: required columns are missing.")

## 12. Delivery time and product ratings

If these fields exist in the data, inspect their relationship visually and summarize the available values. Check for invalid values (for example, negative delivery days or ratings outside the expected scale) before interpreting results.

In [ ]:
if all(c in train.columns for c in ["Delivery_Days", "Product_Rating"]):
    display(train[["Delivery_Days", "Product_Rating"]].describe().T)

    plot_data = train[["Delivery_Days", "Product_Rating"]].dropna()
    plot_data = plot_data.sample(n=min(4000, len(plot_data)), random_state=42)

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=plot_data, x="Delivery_Days", y="Product_Rating", alpha=0.4)
    plt.title("Product Rating vs Delivery Days (Sample)")
    plt.tight_layout()
    plt.show()
else:
    print("Delivery/rating analysis skipped: one or both columns are missing.")

## 13. Executive summary — generate key metrics

This section creates a compact set of metrics that can be used in a portfolio write-up. Values are calculated from the loaded file; do not copy example numbers from elsewhere.

In [ ]:
if "Purchase" in train.columns:
    purchase = pd.to_numeric(train["Purchase"], errors="coerce")
    print("PROJECT SNAPSHOT")
    print("-" * 40)
    print(f"Rows analysed: {len(train):,}")
    print(f"Valid purchase records: {purchase.notna().sum():,}")
    print(f"Total purchase value: {purchase.sum():,.2f}")
    print(f"Average purchase value: {purchase.mean():,.2f}")
    print(f"Median purchase value: {purchase.median():,.2f}")

    if "Product_Category_1" in train.columns:
        top_category = train.groupby("Product_Category_1")["Purchase"].sum().idxmax()
        print(f"Category with highest total purchase value: {top_category}")

    if "City_Category" in train.columns:
        city_avg = train.groupby("City_Category")["Purchase"].mean().sort_values(ascending=False)
        print(f"City category with highest average purchase: {city_avg.index[0]} ({city_avg.iloc[0]:,.2f})")
else:
    print("Cannot generate purchase metrics because 'Purchase' is not available.")

## 14. Conclusions and business recommendations

Complete this section after running the notebook. Replace the prompts below with findings supported by your actual results.

1. **Customer segments:** Which segment has the highest average and median purchase? Include group sizes.
2. **Product categories:** Which category has the highest total purchase value, and which has the highest average?
3. **Geography:** Which city category contributes most to total purchase value?
4. **Data quality:** Which columns have missing values or suspicious values, and what should be done?
5. **Business action:** Suggest one action supported by the analysis, and state what additional data would help validate it.

### Limitations
- The notebook describes patterns in the available dataset; it does not establish causation.
- Confirm the dataset's source, definitions, currency, time period, and whether each row represents an order, item, or customer before publishing business conclusions.
- If this project is used for a predictive model, keep model development separate from exploratory analysis and document the target variable, baseline, validation strategy, and evaluation metrics.

## 15. Portfolio-ready project description

**Flipkart E-commerce Purchase Analysis | Python, Pandas, NumPy, Matplotlib, Seaborn**

Analysed e-commerce transaction data to assess purchase-value distributions, customer-segment patterns, product-category performance, and data quality. Built reusable visualizations and grouped summaries to communicate findings and identify potential business opportunities. Finalize the description with verified dataset size and 2–3 quantified findings after running the notebook.